# 116_paper_figures_author_strict_colab

Build Figures 2-14 using the strict author pipeline (`scripts/build_paper_figures_author_strict.py`).

This notebook is intended for Colab and local Jupyter runs.

In [ ]:
import os
import sys
import shlex
import pathlib
import subprocess

import torch
from IPython.display import Image, display


def _find_project_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir() and (p / "scripts" / "build_paper_figures_author_strict.py").is_file():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir() and (cand / "scripts" / "build_paper_figures_author_strict.py").is_file():
        return cand
    raise RuntimeError(
        "Project root not found. On Colab, clone repo first, for example:\n"
        "!cd /content && git clone https://github.com/codist-posist/econml.git"
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
# Run configuration
ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts")
FIG_DIR = str(PROJECT_ROOT / "artifacts" / "paper_figures")

# "author" matches author code; "paper" uses paper-style flex/output-gap mode.
CONS_MODE = "author"

USE_SELECTED = True
ENSURE_POSTPROCESS = True
ENSURE_IR = True

# Set both to True to force a clean rebuild of figure inputs.
FORCE_REBUILD_POSTPROCESS = True
FORCE_REBUILD_IR = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float64"

PRE = 5
N_POST = 20
IR_H = 120
SHOCK_SIZE_MULT = 1.5

print("CONS_MODE:", CONS_MODE)
print("DEVICE:", DEVICE)
print("FIG_DIR:", FIG_DIR)


In [ ]:
cmd = [
    sys.executable,
    "scripts/build_paper_figures_author_strict.py",
    "--artifacts-root", ARTIFACTS_ROOT,
    "--fig-dir", FIG_DIR,
    "--device", DEVICE,
    "--dtype", DTYPE,
    "--use-selected", str(USE_SELECTED).lower(),
    "--ensure-postprocess", str(ENSURE_POSTPROCESS).lower(),
    "--ensure-ir", str(ENSURE_IR).lower(),
    "--cons-mode", CONS_MODE,
    "--force-rebuild-postprocess", str(FORCE_REBUILD_POSTPROCESS).lower(),
    "--force-rebuild-ir", str(FORCE_REBUILD_IR).lower(),
    "--pre", str(PRE),
    "--n-post", str(N_POST),
    "--ir-h", str(IR_H),
    "--shock-size-mult", str(SHOCK_SIZE_MULT),
]

print("Running command:\n", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)
print("Done.")


In [ ]:
fig_dir = pathlib.Path(FIG_DIR)
fig_paths = [fig_dir / f"figure{i}.png" for i in range(2, 15)]
missing = [p for p in fig_paths if not p.exists()]

print("Figure directory:", fig_dir)
if missing:
    print("Missing files:")
    for p in missing:
        print(" -", p.name)
else:
    print("All Figure 2-14 files found.")

for p in fig_paths:
    if p.exists():
        print("\n", p.name)
        display(Image(filename=str(p)))


In [ ]:
note = pathlib.Path(FIG_DIR) / "figure1_note.txt"
if note.exists():
    print(note.read_text(encoding="utf-8"))
else:
    print("figure1_note.txt not found")
